# pw01 — Sionna 로 ISAC 센싱을 한 선행 논문들

> ⚠ **이 노트북은 생성물이다. 수정은 `prior_work/src/make_pw01.py` 에서** 하고 재실행할 것.
> 모든 사실·인용은 `prior_work/outputs/prior_work.json`(2× 딥리서치 + 직접 웹확인) 에서 주입한다.

**두 질문에 답한다:** ① *Sionna 로 ISAC(센싱)을 한 선행 논문이 실제로 존재하나?* 
② 존재한다면 *우리가 마주한 간극(스톡 Sionna 는 소형 표적의 코히어런트 RCS 를 메쉬에서 못 낸다, report06)을 그들은 어떻게 우회했나?*

**검증 등급(정직성 장치):** ✅검증=CONFIRMED · 🔎직접확인=WEB · 📄단일출처=SOURCE · ⚠미검증단서=IMAGE_LEAD
— 다른 AI 답변에 나온 논문명은 환각 가능성이 있어, **실재를 1차 출처로 확인한 것만** 싣는다.

## §1. 한 줄 답

**① 존재한다 — 다수.** 아래 6건은 전부 실재를 확인했다(arXiv/GitHub 1차 출처). 
**② 소형 표적 코히어런트 RCS 를 스톡 Sionna 로 푼 선행은 없다.** 표준 우회는 세 갈래다:

| 우회 | 뜻 | 대표 선행 |
|---|---|---|
| **(b) 확산 산란계수 S** | 표면에 S∈[0,1] 를 주고 산란전력을 S² 로 배분(재질별, 실측 보정) | Great-X, Deterministic-Modeling |
| **(c) RCS 점표적 주입** | 표적을 점으로 보고 σ(상수)를 채널에 넣음 (h = h_bg + h_target) | NIST 5GNRad·3GPP·MATLAB(→pw02) |
| **(d) 커스텀 산란 add-on** | Sionna 를 기저엔진으로만 쓰고 산란모델을 직접 얹음 | Ziganshin(UTD 회절), **우리(SBR+PO)** |

우리는 (d)로 **값을 계산**(SBR+PO)해 (c)로 **채널에 주입**한다 — 뒤에서 자세히(§4·pw03).

---
## §2. Sionna 로 드론/차량을 센싱한 선행 (표적 산란이 핵심)

### ✅검증  Unreal is all you need: Multimodal ISAC Data Simulation with Only One Engine
- **저자·출처**: Kongwu Huang, Shiyi Mu, Jun Jiang, Yuan Gao (Shanghai Univ) · Shugong Xu (XJTLU) — arXiv:2507.08716 (2025-07, cs.CV) · 플랫폼명 Great-X  ([1](https://arxiv.org/abs/2507.08716) · [2](https://arxiv.org/html/2507.08716v3))
- **무엇을 센싱?** 소형 저고도 UAV(드론) — 상하이 루자쭈이 상공 비행, Great-MSD 데이터셋(CSI+RGB+Radar+LiDAR)
- **표적 산란 처리**: b) 확산 산란계수 S∈[0,1] (S² 확산 + 경면). Sionna 식 광선추적을 Unreal 안에 재구현, Sionna RT 로 교차검증
- **검출체인?** 없음 — CFAR/거리-도플러 아님. CSI 기반 UAV 3D 위치추정
- **우리와의 관계**: 드론 RCS 를 메쉬에서 코히어런트 계산하지 않음(확산계수 사용) → 우리 주장과 모순 없음. 드론을 '산란 표적'이 아니라 '통신/센싱 플랫폼'으로 다룸

### 📄단일출처  Deterministic Modeling of Dynamic ISAC Channels in RF Digital Twin Environments
- **저자·출처**: Cesar Montaner, Saúl Fenollosa, Andres Ortega, Hugo Beltrán, Narcis Cardona (iTEAM, U. Politècnica de València) — arXiv:2603.28736 · EuCAP 2026 채택  ([1](https://arxiv.org/abs/2603.28736))
- **무엇을 센싱?** 실차량 — Nissan Micra(모노스태틱), KIA Xceed(바이스태틱, UE 탑재). 드론/소형 표적 없음
- **표적 산란 처리**: b) Sionna 내장 확산 산란(R²+S²=1), 재질별 S 를 79GHz E-band 채널 사운딩 실측으로 보정
- **검출체인?** 채널 레벨(디지털 트윈 채널 생성)
- **우리와의 관계**: ⭐ **우리 주장 직접 지지** — 논문 스스로 '메쉬 위 순수 경면 광선추적은 mmWave 산란전력을 과소평가하고, 정확한 EM 산란에 필요한 λ/10 메싱은 계산상 불가능'이라고 명시. = report06 의 핵심 주장

### 📄단일출처  Ray-Based Simulation of Scattering from Discretized Curved Bodies for Vehicular and ISAC Applications
- **저자·출처**: Ainur Ziganshin, Enrico M. Vitucci, Wim Kotterman, Reiner Thomä, Christian Schneider, Vittorio Degli-Esposti — arXiv:2604.05991 (eess.SP, 2026-04)  ([1](https://arxiv.org/pdf/2604.05991))
- **무엇을 센싱?** 정준 곡면체(구·반경 7λ 원통) + 실차량 i-MiEV(1496/220 facet) @2-3GHz. 소형/서브미터 표적 없음
- **표적 산란 처리**: d) **커스텀 add-on** — Sionna-RT v0.19 를 기저 엔진으로만 쓰고, 저자들이 UTD 회절(엣지·꼭짓점·이중엣지) 확장을 직접 구현. 경면 facet + UTD. **PO/SBR 은 관련연구로 인용만 하고 안 씀**
- **검출체인?** 해석해·FEKO MLFMM·BIRA 실측 대조 검증
- **우리와의 관계**: 'Sionna 에 커스텀 산란을 더한다'는 우리와 같은 정신. 단 방법은 UTD(우리는 PO/SBR), 표적은 대형

> 🔑 **읽는 법.** 셋 다 '표적을 메쉬에서 코히어런트 RCS 로 계산'하지 **않는다**. Great-X·Deterministic-Modeling 은 **확산계수 S**(b)로, Ziganshin 은 **커스텀 UTD**(d)로 우회한다. 우리와 정신이 가장 가까운 건 Ziganshin(‘Sionna + 직접 만든 산란’)이지만, 그들은 UTD·대형표적, 우리는 PO/SBR·소형 드론이다.

---
## §3. Sionna 를 ISAC 에 쓰지만 '표적 RCS'는 안 하는 선행 (문맥용)

### 📄단일출처  CISSIR: Beam Codebooks with Self-Interference Reduction Guarantees for ISAC Beyond 5G
- **저자·출처**: Rodrigo Hernangómez, Jochen Fink, Renato L. G. Cavalcante, Sławomir Stańczak — arXiv:2502.10371 (2025) · **NVIDIA 공식 'Made with Sionna' 유일 ISAC 등재** · Sionna v0.17 · code github.com/rodrihgh/cissir  ([1](https://nvlabs.github.io/sionna/made_with_sionna.html) · [2](https://arxiv.org/abs/2502.10371))
- **무엇을 센싱?** 물리 표적 없음 — 자기간섭 저감·양자화잡음 한계로 센싱 성능만 다룸
- **표적 산란 처리**: n/a
- **검출체인?** n/a
- **우리와의 관계**: 스톡 Sionna 를 표적 RCS 에 쓰지 않음 — 우리 주장과 정합. NVIDIA 공식 쇼케이스에도 소형표적 RCS ISAC 이 없다는 방증

### ✅검증  SimART: A Unified and Open Real-world Multimodal Simulation Platform for 6G ISAC
- **저자·출처**: Kang Yan, Yuqi Cao, Jiaqi Li, Luping Xiang, Kun Yang (UESTC, Nanjing Univ) — arXiv:2605.13309 (2026-05) · github.com/guchuanv-alt/SimART (~168★)  ([1](https://arxiv.org/abs/2605.13309) · [2](https://github.com/guchuanv-alt/SimART))
- **무엇을 센싱?** UAV 를 **비전(YOLOv8 on RGB)+GPS** 로 센싱(빔예측용). 레이더 에코 아님
- **표적 산란 처리**: 환경(건물)만 Sionna RT 경면+회절, 미세형상은 확산 미미 이유로 폐기. 표적 산란모델 없음
- **검출체인?** 레이더 검출 없음 — 'RCS/radar/CFAR' 문자열이 논문에 전무
- **우리와의 관계**: 다른 AI 이미지가 'Sionna 차용 ISAC'으로 든 예시지만, 실제로는 통신+온보드 다중센서 플랫폼. 우리 패시브 레이더 조각(RCS/검출) 없음

> **왜 이 둘도 싣나.** CISSIR 은 **NVIDIA 공식 'Made with Sionna' 쇼케이스의 유일한 ISAC 등재작**인데도 물리 표적 RCS 를 다루지 않는다 — 소형표적 RCS ISAC 이 스톡 Sionna 의 표준 용례가 아니라는 방증이다. SimART 는 다른 AI 답변이 'Sionna ISAC'으로 든 예지만, 실제 '센싱'은 카메라(YOLOv8)+GPS 라 우리 패시브 레이더 조각과 겹치지 않는다(정직한 구분).

---
## §4. 우리 주장은 선행에 의해 **지지**된다

### 📄단일출처  Sionna RT: Differentiable Ray Tracing for Radio Propagation Modeling
- **저자·출처**: Hoydis et al. (NVIDIA) — arXiv:2303.11103 (2023, v0.14~) — Sionna RT 창설 논문  ([1](https://arxiv.org/abs/2303.11103))
- **무엇을 센싱?** n/a (엔진 논문)
- **표적 산란 처리**: 솔버 아키텍처를 문서화: 경면(이미지법 + Fibonacci 격자 광선탐색)·확산(계수 파라미터)·1차 회절. **경로별 복소이득을 반환하지, 표적 표면 위 SBR 장 적분이 아님**
- **검출체인?** n/a — ISAC 은 동기부여 주제로만 언급
- **우리와의 관계**: ⭐ **우리 주장의 1차 근거** — 스톡 솔버가 per-path gain 만 낸다는 것을 창설 논문이 문서화

**우리 report06 주장은 선행에 의해 지지됨: Deterministic-Modeling(EuCAP)이 'λ/10 메싱 불가·경면만이면 산란 과소평가'를 명시, Sionna RT 창설논문이 'per-path gain 반환'을 문서화.**

즉 report06 이 다섯 방식으로 실증한 'PathSolver 에 산란적분 없음'은 우리만의 발견이 아니라, **Sionna 창설 논문이 문서화**하고 **EuCAP 2026 논문이 재확인**한 사실이다. 우리가 한 일은 그 한계를 인정하고 **SBR+PO 를 부분적으로 더한 것**(report07)이다.

In [ ]:
# 이 리포트가 인용한 선행 논문 — 전부 prior_work.json 에서 (손으로 안 적음)
import json, os
J = json.load(open('outputs/prior_work.json', encoding='utf-8'))
for p in J['papers']:
    print(f"[{p['grade']:9s}] {p['venue']}")
    print(f"           {p['title'][:70]}")

---
## §5. 정리

1. **Sionna-ISAC 선행은 많다** — 드론(Great-X)·차량(Deterministic-Modeling·Ziganshin)·NVIDIA 공식(CISSIR)까지.
2. **소형표적 코히어런트 RCS-from-mesh 를 스톡 Sionna 로 푼 선행은 **없다**. 표준 우회 3가지: (b)확산계수 S[Great-X·Det-Modeling] (c)RCS 점표적 주입 h=h_bg+h_target[NIST 5GNRad·3GPP·MATLAB] (d)커스텀 산란 add-on[Ziganshin UTD, 우리 SBR+PO].**
3. 우리 위치와 '덜 점유된 틈새'는 다음 편에서 — **도구 지도(pw02)** 와 **포지셔닝(pw03)**.

> **다음** → [pw02 — 오픈소스 센싱/ISAC 도구 지도](pw02_opensource_tools.ipynb): NIST 5GNRad·RadarSimPy·OpenISAC 가 무엇을 해 주고, 우리 프로젝트에 무엇을 채택할까.